In [1]:
# importing libraries

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import duckdb
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split
import math
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [2]:
# importing the sessionized df

session_df = pd.read_pickle("data/intermediate_files/sessionized_events.pkl")
item_prop_df = pd.read_pickle("data/intermediate_files/item_properties.pkl")

In [3]:
item_cat_map_query = """
select
    itemid, value as category_id, row_number() over(partition by itemid order by timestamp desc) as rn
from item_prop_df
where
    property = 'categoryid'
qualify rn = 1
"""

item_cat_map_df = duckdb.sql(item_cat_map_query).to_df()

In [4]:
session_df = pd.merge(left=session_df, right=item_cat_map_df[['itemid', 'category_id']], on='itemid', how='left')

mode_cat = item_cat_map_df.category_id.mode()[0]
session_df.category_id = session_df.category_id.fillna(mode_cat)

In [5]:
# splitting for train and test time-wise
session_train, session_test = train_test_split(session_df, test_size=0.2, shuffle=False)

# we will only take new session entries as valid queries to replicate app open
new_session = ((session_test.is_new_session == 1) & (session_test.session_id > 1))

# this consists 50000 events
sample_rows = 50000
query_base = session_test.loc[new_session].sample(n=min(sample_rows,session_test.shape[0]))

In [6]:
# query table for past data

def build_query_table_past(query_df, session_df, k=25, C=5):

    output_rows = []

    # group once for speed
    user_groups = {
        user_id: grp.sort_values("sequence_no").reset_index(drop=True)
        for user_id, grp in session_df.groupby("visitorid")
    }

    for query_id, (_, row) in enumerate(query_df.iterrows()):

        udf = user_groups[row["visitorid"]]

        hist_df = udf[udf.timestamp < row['timestamp']]

        last_k_items = set(hist_df['itemid'].drop_duplicates().tail(k).tolist())
        last_C_categories = set(hist_df['category_id'].drop_duplicates().tail(C).tolist())

        event_grouped = hist_df.groupby("event")

        past_events = event_grouped.size()

        # purchased items
        pre_purchased = set(event_grouped['itemid'].apply(list).get('transaction',[]))

        output_rows.append({
            "query_id": query_id,
            "visitorid": row["visitorid"],
            "session_id": row["session_id"],
            "sequence_no": row["sequence_no"],
            "event_pos_in_session": row["event_pos_in_session"],
            "query_time": row["timestamp"],
            "current_event": row["event"],
            "last_k_items": last_k_items,
            "last_C_categories": last_C_categories,
            "pre_purchased": pre_purchased,
            "past_view_count": past_events.get("view", 0),
            "past_atc_count": past_events.get("addtocart", 0),
            "past_order_count": past_events.get("transaction", 0)
        })

    return pd.DataFrame(output_rows)

In [7]:
def build_query_table_fut(query_df, session_fut_df):

    output_rows = []

    # group once for speed - past data
    user_groups = {
        user_id: grp.sort_values("sequence_no").reset_index(drop=True)
        for user_id, grp in session_fut_df.groupby("visitorid")
    }

    for query_id, (_, row) in enumerate(query_df.iterrows()):

        user_df = user_groups[row["visitorid"]]

        curr_sess = row["session_id"]
        curr_ts = row["timestamp"]

        # future positives within same session, using only all future interactions in the same session
        fut_df = user_df[
            (user_df["session_id"] == curr_sess) &
            (user_df["timestamp"] >= curr_ts)
            # & (user_df["event"] == "view")
        ]

        future_positives = list(dict.fromkeys(fut_df["itemid"].tolist()))

        if len(future_positives) == 0:
            continue

        output_rows.append({
            "query_id": query_id,
            "future_positives": future_positives
        })

    return pd.DataFrame(output_rows)

In [8]:
# remove duplicates basis ('visitorid','session_id','timestamp') so that we keep only one entry per combination for testing
query_base.drop_duplicates(subset=['visitorid','session_id','timestamp'],inplace=True)

query_ip_past = build_query_table_past(query_base, session_df) #for building past based inputs
query_ip_fut = build_query_table_fut(query_base, session_test) #for building future positives

#merging togethor
query_input = pd.merge(left=query_ip_past, right=query_ip_fut, on='query_id', how='inner')

#### User Agnostic CG feeders

In [9]:
## platform popularity based on orders basis 7 and 30 day popularity

def popularity_cg_base(size, session_past_df, d):

    max_time = session_past_df.timestamp.max()
    min_time = max_time - pd.DateOffset(days=d)

    orders_past_df = session_past_df.loc[((session_past_df.transactionid.notnull()) & (session_past_df.timestamp >= min_time))]

    return orders_past_df.groupby('itemid').size().sort_values(ascending=False).index.tolist()[:size]

In [10]:
## platform category popularity based on last 30 days

def category_cg_base(session_past_df, d, cat_rank, item_rank):

    max_time = session_past_df.timestamp.max()
    min_time = max_time - pd.DateOffset(days=d)

    orders_past_df = session_past_df.loc[((session_past_df.transactionid.notnull()) & (session_past_df.timestamp >= min_time))]
    
    cat_rank_query = f"""
    select 
        * 
        , row_number() over(partition by category_id order by total_orders desc) as item_rank
        , dense_rank() over(order by cat_orders desc) as cat_rank
    from (
    select 
        *, sum(total_orders) over(partition by category_id) as cat_orders from (
    select
        category_id, itemid, count(*) as total_orders
    from orders_past_df
    group by 1,2
    ) t1 ) t2
    order by cat_rank, item_rank
    """

    cat_rank_df = duckdb.sql(cat_rank_query).to_df()

    return cat_rank_df.loc[((cat_rank_df.cat_rank <= cat_rank) & (cat_rank_df.item_rank <= item_rank)),'itemid'].tolist(), cat_rank_df

In [11]:
## scores item basis their spread of users, interactions

def generality_CG(session_past_df, d, size):
    
    max_time = session_past_df.timestamp.max()
    min_time = max_time - pd.DateOffset(days=d)

    hist_df = session_past_df[(session_past_df.timestamp >= min_time)]

    generality_query = """
    select
        category_id
        , itemid

        , count(distinct visitorid) as unique_users
        , count(distinct session_id) as unique_sessions
        , count(distinct date(timestamp)) as unique_days

        , (
            0.1*sum(case when event='view' then 1 else 0 end) 
            + 0.3*sum(case when event='addtocart' then 1 else 0 end) 
            + 0.6*sum(case when event='transaction' then 1 else 0 end)
        ) as weighted_interaction_score

    from hist_df
    group by 1, 2
    """

    item_stats = duckdb.sql(generality_query).to_df()

    for col in ['unique_users', 'unique_sessions', 'unique_days', 'weighted_interaction_score']:

        item_stats[f'{col}_pct'] = item_stats[col].rank(pct=True)

    item_stats['generality_score'] = (
        0.5*item_stats['weighted_interaction_score_pct']
        + 0.25*item_stats['unique_users_pct']
        + 0.15*item_stats['unique_sessions_pct']
        + 0.10*item_stats['unique_days_pct']
    )

    item_stats = duckdb.sql('select *, row_number() over(partition by category_id order by generality_score desc) as item_rank from item_stats').to_df()

    return item_stats.sort_values('generality_score', ascending=False)['itemid'].to_list()[:size], item_stats



#### User based CG feeders

In [12]:
## top items based on categories interacted by the user

def user_category_cg_base(input_q, cat_stats_df, item_rank):

    past_cats = input_q['last_C_categories']

    if not past_cats:
        return []

    return cat_stats_df.loc[(
        (cat_stats_df.category_id.isin(past_cats)) & (cat_stats_df.item_rank <= item_rank)
        ), 'itemid'].tolist()


In [13]:
def user_generality_cg_base(input_q, general_df, item_rank):

    past_cats = input_q['last_C_categories']

    if not past_cats:
        return []
    
    return general_df.loc[((general_df.category_id.isin(past_cats)) & (general_df.item_rank <= item_rank)), 'itemid'].tolist()

In [14]:
## Co - interaction CG

# event_type : orders - returns visitorid and items that user bought over last 90 days - for co-order
# event_type : interactions - returns visitorid, session id, min timestamp and items in that session in list over last 30 days - for co-interacted

def build_event_item_table(events_df, event_type, d):

    max_time = events_df.timestamp.max()
    min_time = max_time - pd.DateOffset(days=d)

    if event_type=='orders':
        group = ["visitorid"]
        condition = (events_df.timestamp >= min_time)
        agg_condition = {
            "items": (
                "itemid",
                lambda x: list(dict.fromkeys(x.tolist()))
            )
        }
    
    else:
        group = ["visitorid", "session_id"]
        condition = ((events_df.timestamp >= min_time) & (events_df["event"].isin(["view", "addtocart","transaction"])))
        agg_condition = {
            "session_start": ("timestamp", "min"),

            "items": (
                "itemid",
                lambda x: list(dict.fromkeys(x.tolist()))
            )
        }

    browse_df = events_df.loc[condition].copy()

    event_items = (
        browse_df.groupby(group)
        .agg(**agg_condition)
        .reset_index()
    )

    return event_items

In [15]:
# item item co-event count stored 

def build_same_event_item_map(event_item_df, event_type, top_m, max_items_per_session=30):

    item_to_related = defaultdict(Counter)

    for items in event_item_df["items"]:
        if len(items) < 2:
            continue

        # optional cap to avoid huge sessions causing too much work
        if event_type == 'interactions' and len(items) > max_items_per_session:
            items = items[:max_items_per_session]

        for i, item in enumerate(items):
            other_items = items[:i] + items[i+1:]
            item_to_related[item].update(other_items)

    # keep only top M related items per item for fast query-time retrieval
    item_to_top_related = {
        item: counter.most_common(top_m)
        for item, counter in item_to_related.items()
    }

    return item_to_top_related

In [16]:
def co_event_cg(input_q, size, item_to_top_related):
    
    '''
    inputs - 
    input_q - pre-purchased list, last k interacted items (click/atc/order), past popular items of platform that the user hasn't bought yet
    size - size of output
    item_to_top_related - co-order map or co-interaction map

    output -
    recommender_list - array of recommended products
    '''
        
    already_purchased = input_q["pre_purchased"]
    seed_items = input_q["last_k_items"]

    if not seed_items:
        return []

    scores = Counter()

    for seed_item in seed_items:
        for related_item, cnt in item_to_top_related.get(seed_item, []):
            if related_item != seed_item and related_item not in already_purchased:
                scores[related_item] += cnt

    return [item for item, _ in scores.most_common(size)]

#### Enabling CG outputs

In [17]:
SOURCE_WEIGHTS = {
    "user_co_interaction": 1.00, "user_category": 0.75, "user_generality": 0.65,
    "global_category_pop_d7": 0.45, "global_category_pop_d30": 0.35,
    "global_pop_d7": 0.30, "global_pop_d30": 0.20, "global_generality_d30": 0.30,
}

def rank_decay_score(rank):
    return 1.0 / np.log2(rank + 1.0)

def dedupe_keep_order(items):
    if items is None: return []
    seen, out = set(), []
    for item in items:
        if pd.isna(item): continue
        item = int(item)
        if item not in seen:
            seen.add(item); out.append(item)
    return out

def add_source_scores(cg_scored, source_name, items, source_weight):
    for rank, item in enumerate(dedupe_keep_order(items), start=1):
        contrib = source_weight * rank_decay_score(rank)
        if item not in cg_scored:
            cg_scored[item] = {"raw_score": 0.0, "num_sources": 0, "sources": set()}
        cg_scored[item]["raw_score"] += contrib
        if source_name not in cg_scored[item]["sources"]:
            cg_scored[item]["num_sources"] += 1
            cg_scored[item]["sources"].add(source_name)

def build_combined_cg_list_for_row(row, pop_items_d7, pop_items_d30, cat_pop_items_d7,
                                   cat_pop_items_d30, gen_pop_items_d30, final_size=500):
    cg_scored = {}

    sources = [
        ("user_co_interaction", row["user_co_interaction_CG"]),
        ("user_category", row["user_cat_CG"]),
        ("user_generality", row["user_generality_CG"]),
        ("global_category_pop_d7", cat_pop_items_d7),
        ("global_category_pop_d30", cat_pop_items_d30),
        ("global_pop_d7", pop_items_d7),
        ("global_pop_d30", pop_items_d30),
        ("global_generality_d30", gen_pop_items_d30),
    ]

    for name, items in sources:
        add_source_scores(cg_scored, name, items, SOURCE_WEIGHTS[name])

    if not cg_scored: return [], [], []

    max_score = max(v["raw_score"] for v in cg_scored.values())
    scored = [
        {
            "itemid": item,
            "raw_score": info["raw_score"],
            "norm_score": info["raw_score"] / max_score if max_score > 0 else 0.0,
            "num_sources": info["num_sources"],
        }
        for item, info in cg_scored.items()
    ]

    scored = sorted(scored, key=lambda x: (-x["norm_score"], -x["raw_score"], -x["num_sources"], x["itemid"]))[:final_size]
    return [x["itemid"] for x in scored], [x["norm_score"] for x in scored], [x["raw_score"] for x in scored]

def combine_CG_for_metrics(query_input, session_train, final_size=500):
    query_input = query_input.copy()

    pop_items_d7 = popularity_cg_base(50, session_past_df=session_train, d=7)
    pop_items_d30 = popularity_cg_base(50, session_past_df=session_train, d=30)

    cat_pop_items_d7, cat_stats_d7_df = category_cg_base(session_past_df=session_train, d=7, cat_rank=5, item_rank=10)
    cat_pop_items_d30, _ = category_cg_base(session_past_df=session_train, d=30, cat_rank=5, item_rank=10)

    gen_pop_items_d30, item_stats_df = generality_CG(session_past_df=session_train, d=30, size=100)

    session_items_df = build_event_item_table(session_train, event_type="interactions", d=30)
    item_to_top_interacted_map = build_same_event_item_map(session_items_df, event_type="interactions", top_m=300, max_items_per_session=30)

    query_input["user_cat_CG"] = query_input.apply(user_category_cg_base, axis=1, cat_stats_df=cat_stats_d7_df, item_rank=20)
    query_input["user_generality_CG"] = query_input.apply(user_generality_cg_base, axis=1, general_df=item_stats_df, item_rank=20)
    query_input["user_co_interaction_CG"] = query_input.apply(co_event_cg, axis=1, size=200, item_to_top_related=item_to_top_interacted_map)

    cg_items, cg_norm_scores, cg_raw_scores = [], [], []
    for _, row in query_input.iterrows():
        items, norm_scores, raw_scores = build_combined_cg_list_for_row(
            row, pop_items_d7, pop_items_d30, cat_pop_items_d7, cat_pop_items_d30, gen_pop_items_d30, final_size
        )
        cg_items.append(items); cg_norm_scores.append(norm_scores); cg_raw_scores.append(raw_scores)

    query_input["combined_CG"] = cg_items
    query_input["combined_CG_norm_scores"] = cg_norm_scores
    query_input["combined_CG_raw_scores"] = cg_raw_scores
    return query_input

In [18]:
final_query_ip = combine_CG_for_metrics(query_input, session_train, final_size=500)

#### CG evaluation

In [19]:
# Evaluation with HitRate, Recall, precision and MRR

def evalualtion_cg_query(true_op, pred_op, k):

    true_op_set, pred_op_set = set(true_op), set(pred_op[:k])

    common_items = true_op_set.intersection(pred_op_set)

    # hitrate
    hit_rate_k = 1 if len(common_items) > 0 else 0
    # recall
    recall_k = len(common_items)/len(true_op_set)
    # precision
    precision_k = len(common_items)/len(pred_op_set)
    #  MRR_k
    MRR_k = 0
    for rank, item in enumerate(pred_op[:k], start=1):
        if item in true_op_set:
            MRR_k = 1/rank
            break

    return {"hitrate_k":hit_rate_k, "recall_k":recall_k, "precision_k":precision_k, "MRR_k":MRR_k}

In [20]:
def evalualtion_cg(eval_df, k):

    metrics = defaultdict(list)
    len_df = eval_df.shape[0]

    for _, row in eval_df.iterrows():
        true_op = row['future_positives']
        pred_op = row['predicted_items']

        res = evalualtion_cg_query(true_op, pred_op, k)

        metrics['hitrate_k'].append(res['hitrate_k'])
        metrics['recall_k'].append(res['recall_k'])
        metrics['precision_k'].append(res['precision_k'])
        metrics['MRR_k'].append(res['MRR_k'])

    return {
        'mean_hitrate_k': sum(metrics['hitrate_k'])/len_df,
        'mean_recall_k': sum(metrics['recall_k'])/len_df,
        'mean_precision_k': sum(metrics['precision_k'])/len_df,
        'mean_MRR_k': sum(metrics['MRR_k'])/len_df
    }

In [21]:
# co-interaction cg output
combined_cg_op_df = pd.DataFrame(
    {'future_positives': final_query_ip.future_positives, 
     'predicted_items': final_query_ip.combined_CG
    })

# Evaluation at differnt k values

k_vals = [100,200,300,500]

for k in k_vals:
    print(f'At k = {k}')
    print("Combined CG evalutaion : ",evalualtion_cg(combined_cg_op_df, k=k))

    print('\n-------------------------------------------------------------------\n')

At k = 100
Combined CG evalutaion :  {'mean_hitrate_k': 0.27032, 'mean_recall_k': 0.23851391310592462, 'mean_precision_k': 0.003299199999999966, 'mean_MRR_k': 0.050197040670937366}

-------------------------------------------------------------------

At k = 200
Combined CG evalutaion :  {'mean_hitrate_k': 0.29626, 'mean_recall_k': 0.2618348711872802, 'mean_precision_k': 0.0018484999999998923, 'mean_MRR_k': 0.05038849081138885}

-------------------------------------------------------------------

At k = 300
Combined CG evalutaion :  {'mean_hitrate_k': 0.30572, 'mean_recall_k': 0.27042155048832067, 'mean_precision_k': 0.0014262913422571442, 'mean_MRR_k': 0.05042811273720614}

-------------------------------------------------------------------

At k = 500
Combined CG evalutaion :  {'mean_hitrate_k': 0.31132, 'mean_recall_k': 0.27455232829510967, 'mean_precision_k': 0.00130211300847554, 'mean_MRR_k': 0.05044382822307143}

-------------------------------------------------------------------


In [24]:
# save the updated session df

session_df.to_pickle('data/intermediate_files/sessionized_events.pkl')

In [22]:
# we will now make a combined CG which will have
# 1. co - interaction CG - last 50 items (size=200)
# 2. item popularity CG - 7 & 30 days (size = 50/50) done - based on orders
# 3. category popularity CG - selects 5 platform best category and 10 items from each (size = 50) - done
# 3. category based on users interaction - top 5 categories of history of user 20 from each category (size=100) - done
# 4. generality aware (size = 100) done
